In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import shap
import pickle

In [2]:
from google.colab import drive
#drive.mount('/content/drive')
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


##Get the overall, Min and Max local SHAP values

In [4]:
species_list = ["BGGN"] # Shrubland

for species in species_list:

  with open(f'/content/drive/My Drive/SHAP_multipredictors/Guild/Woodland/4x4/SHAP/{species}_8Predictors.pkl', 'rb') as f:
      shap_values = pickle.load(f)
shap_df = pd.DataFrame(shap_values)
shap_df[1:10]

,0,1,2,3,4,5,6,7
1,-0.015793,0.0,0.018361,-0.077510,0.0,-0.072463,-0.010165,0.454597
2,-0.015793,0.0,0.018361,-0.077510,0.0,-0.072463,-0.010165,0.454597
3,-0.042936,0.0,0.024496,-0.100142,0.0,-0.137288,-0.026248,0.356199
4,-0.042936,0.0,0.024496,-0.100142,0.0,-0.137288,-0.026248,0.356199
5,-0.000521,0.0,0.025192,0.003640,0.0,0.024785,-0.037626,0.014316
6,-0.042936,0.0,0.024496,-0.100142,0.0,-0.137288,-0.026248,0.356199
7,-0.042936,0.0,0.024496,-0.100142,0.0,-0.137288,-0.026248,0.356199
8,-0.042936,0.0,0.024496,-0.100142,0.0,-0.137288,-0.026248,0.356199
9,0.003365,0.0,0.017764,-0.004633,0.0,-0.008537,-0.070760,0.011525


In [ ]:
#species_list = ["HAFL","OSFL","RCKI","WETA","YRWA"]  #Forest
#species_list = ["BGGN","CAKI","EABL","PLVI","SAPH"] # Woodland
#species_list = ["GRSP", "LBCU", "MOBL", "UPSA", "WEME"]  # Grassland
species_list = ["GRFL","LOSH","SATH","WEKI","YEWA"] # Shrubland

for species in species_list:

  with open(f'/content/drive/My Drive/SHAP_multipredictors/Guild/Shrubland/4x4/SHAP/{species}_8Predictors.pkl', 'rb') as f:
      shap_values = pickle.load(f)

  # Manually name the 36 columns
  column_names = [
      'CumDHI', 'DEM', '%Forest', '%Grassland', 'Precipitation', '%Shrubland', 'Temperature', 'VarDHI' #,
    #'DEM_1024x1024', 'DEM_128x128', 'DEM_16x16', 'DEM_256x256', 'DEM_32x32', 'DEM_4x4', 'DEM_512x512', 'DEM_64x64', 'DEM_8x8',
   # 'minDHI_1024x1024', 'minDHI_128x128', 'minDHI_16x16', 'minDHI_256x256', 'minDHI_32x32', 'minDHI_4x4', 'minDHI_512x512', 'minDHI_64x64', 'minDHI_8x8',
    #'varDHI_1024x1024', 'varDHI_128x128', 'varDHI_16x16', 'varDHI_256x256', 'varDHI_32x32', 'varDHI_4x4', 'varDHI_512x512', 'varDHI_64x64', 'varDHI_8x8'
  ]

  # Convert SHAP values to a DataFrame for convenience
  shap_df = pd.DataFrame(shap_values, columns=column_names) #shap_values[:,:,1]

  #preparing to combine uniqueID, lat and long with the respective shap values
  data = pd.read_csv(f"/content/drive/My Drive/SHAP_multipredictors/Guild/Shrubland/4x4/Raster_extracted_4x4_Raster_extracted_Pred_ {species} .csv")
  selected_columns = data[['uniqueID', 'PointLatit', 'PointLongi', 'Predicted']]

  # Concatenate the selected columns with the shap_df
  combined_df = pd.concat([shap_df, selected_columns], axis=1)
  combined_df.to_excel(f'/content/drive/My Drive/SHAP_multipredictors/Guild/Shrubland/4x4/SHAP/shp/Mulipredictors_shap_values_{species}.xlsx', index=False)
  # Display the combined DataFrame
  #print(combined_df.head())


  # Find the maximum value in each row and its column name
  max_values = shap_df.max(axis=1)
  max_columns = shap_df.idxmax(axis=1)

  min_values = shap_df.min(axis=1)
  min_columns = shap_df.idxmin(axis=1)
  # Combine results into a DataFrame
  result = pd.DataFrame({
      'Max_Value': max_values,
      'Max_Column': max_columns,
      'Min_Value': min_values,
      'Min_Column': min_columns
  })

  result_df = pd.concat([result, selected_columns], axis=1)
  # Display the first few rows of the result
  #print(result_df.head())
  result_df.to_excel(f'/content/drive/My Drive/SHAP_multipredictors/Guild/Shrubland/4x4/SHAP/shp/Mulipredictors_MinMax_SHAP_{species}.xlsx', index=False)

## Making Shapefiles

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

species_list = ["HAFL","OSFL","RCKI","WETA","YRWA"]  # Forest
#species_list = ["BGGN","CAKI","EABL","PLVI","SAPH"] # Woodland
#species_list = ["GRSP", "LBCU", "MOBL", "UPSA", "WEME"]  # Grassland
#species_list = ["GRFL","LOSH","SATH","WEKI","YEWA"] # Shrubland

for species in species_list:

  # Step 1: Read Excel file
  excel_file = f'/content/drive/My Drive/SHAP_multipredictors/Guild/Forest/4x4/SHAP/shp/Excel/Mulipredictors_MinMax_SHAP_{species}.xlsx'  # Change this to your file path
  df = pd.read_excel(excel_file)

  # Step 2: Ensure the dataset has latitude & longitude columns
  latitude_col = "PointLatit"  # Change to the actual column name in your Excel
  longitude_col = "PointLongi"  # Change to the actual column name in your Excel

  if latitude_col not in df.columns or longitude_col not in df.columns:
      raise ValueError("Latitude or Longitude column not found in the dataset")

  # Step 3: Convert to GeoDataFrame
  geometry = [Point(xy) for xy in zip(df[longitude_col], df[latitude_col])]
  gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # WGS 84 CRS

  # Step 4: Save as Shapefile
  output_shapefile = f"/content/drive/My Drive/SHAP_multipredictors/Guild/Forest/4x4/SHAP/shp/MinMax_SHAP_{species}.shp"
  gdf.to_file(output_shapefile, driver='ESRI Shapefile')

#print(f"Shapefile saved as {output_shapefile}")